In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv("./powerplant_data.csv");

In [ ]:
data.head()

# AT->Temp
# V->Vacuum
# AP->pres
# RH->Humidity
# PE ->produce Energy

In [ ]:
data.isnull().sum()

In [ ]:
X = data.drop("PE",axis=1)
y = data["PE"]

In [ ]:
X.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
data.shape

In [ ]:
scaler = StandardScaler();
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
import torch
import torch.nn as  nn

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor =torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [ ]:
# type(y_train)
# type(X_train_scaled)
# view(-1,-1)-to convert shape into [[1],[2],[3] ]

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size = 32)

### Deep Learning

In [ ]:
#Define ANN model

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        
        self.model = nn.Sequential(
            #1st Hidden layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            #2nd Layer
            nn.Linear(6,6),
            nn.ReLU(),

            #output
            nn.Linear(6,1),
        )
    def forward(self,x):
        return self.model(x)
            

In [ ]:
import torch.optim as optim
model = ANN()

#loss,optimizer
criterion = nn.MSELoss();
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train the ANN
epochs = 100;
train_losses=[]
test_losses=[]
best_val_loss = float("inf");
for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # training loss fro 1 epoch
    for xb,yb in train_loader:
        #xb -> features of one batch
        #yb -> label of one batch
        optimizer.zero_grad();
        outputs = model(xb) # predicted o/p for this batch
        loss = criterion(outputs,yb) # compute loss
        loss.backward() # back prop .. compute gradients 
        optimizer.step() # params update

        running_loss +=loss.item(); # converting loss tensor => py float
    epoch_train_loss = running_loss / len(train_loader) # avg loss per batch = train loss of one epoch
    train_losses.append(epoch_train_loss)

    # Validation
    running_val_loss = 0.0
    model.eval()
    with torch.no_grad(): # no compute of gradients - bco torch as auot grad which compute the gradients 
        for xb,yb in test_loader:
            outputs = model(xb);
            loss = criterion(outputs,yb)
            running_val_loss+=loss.item()
        
        epoch_val_loss = running_val_loss / len(test_loader)
        test_losses.append(epoch_val_loss)

        print(f"epoch {epoch+1}/{epochs} => train loss = {epoch_train_loss} & val loss= {epoch_val_loss}")
        if epoch_val_loss < best_val_loss :
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(),"best_model.pt")

In [ ]:
import matplotlib.pyplot as plt
loss_df = pd.DataFrame({
    "Training_Loss":train_losses,
    "Validation_Loss":test_losses
})
plt.plot(loss_df["Training_Loss"],label="Training_Loss");
plt.plot(loss_df["Validation_Loss"],label="Validation_Loss");
plt.xlabel("Epochs");
plt.ylabel("Lossess");
plt.legend();

In [ ]:
#Loading best model
model.load_state_dict(torch.load("best_model.pt"))

In [ ]:
import torch

state_dict = torch.load("best_model.pt")
print(state_dict)

In [ ]:
# Evaluation
model.eval();
with torch.no_grad():
    train_preds = model(X_train_tensor)
    test_preds = model(X_test_tensor)

    train_mse_loss = criterion(train_preds,y_train_tensor);
    test_mse_loss = criterion(test_preds,y_test_tensor);
print("Training MSE :",train_mse_loss.item());
print("Testing MSE :",test_mse_loss.item());

In [ ]:
from sklearn.metrics import r2_score
print("r2_score=",r2_score(y_test,test_preds))

In [ ]:
predicted_df = pd.DataFrame(test_preds.numpy(), columns=["Predicted Values"])
actual_df = pd.DataFrame(y_test.values, columns=["Actual Values"])

pd.concat([predicted_df, actual_df], axis=1)